# FXS / 05 — EURUSD-02 on the sealed 2026 data: one look

Protocol: [`003_claude_fxs05_eurusd02_holdout_preregistration.md`](../../../discussions/FX_Intraday_Seasonality/003_claude_fxs05_eurusd02_holdout_preregistration.md).

**Rule (frozen):** every trading day, news days included, sell EURUSD at the bid at 04:00 New York, buy back at the ask at 08:00 New York. Cost: quoted spread + $3 per lot round trip.

| Test | Condition |
|---|---|
| H1 | 2026 mean net pips > 0 |
| H2 | 2026 mean net significantly below +1.53 pips (one-sided p ≤ 0.05) |
| H3 (reported) | 2026 mean net > 0, one-sided p ≤ 0.05 (≈ 27% power) |

`HOLDS` = H1 and not H2 · `INCONCLUSIVE` = not H1 and not H2 · `FAILS` = H2.

**Run this notebook once.** Commit `EURUSD/03/` without changing anything.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PAIR, CID = "EURUSD", "EURUSD-02"
START, END = dt.date(2026, 1, 1), dt.date(2026, 11, 6)
PROMISED_PIPS = 1.53            # 2021-2025 net pips per trade at $3/lot (FXS 04)
COMMISSION_PER_LOT = 3.0
E0 = 6_000.0

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs05_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from scipy.stats import norm

cfg = F.FXSConfig()
OUT = Path.cwd() / ("03_synthetic" if SYNTHETIC else "03"); OUT.mkdir(exist_ok=True)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    F.write_synthetic_candles(Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy", PAIR, dt.date(2025, 12, 28), END,
                              seed=2026, drift_bps=-3.0, drift_ny=(4, 8))
    cand = pd.DataFrame([{"id": CID, "pair": PAIR, "clock": "NY", "start": 660, "end": 900, "direction": -1}])
else:
    disc = json.loads((Path("01") / "summary.json").read_text())
    assert F.frozen_hash_ok(Path("01") / "candidates_frozen.csv", disc["candidates_sha256"]), "frozen file changed"
    cand = pd.read_csv(Path("01") / "candidates_frozen.csv").query("id == @CID")
r = cand.iloc[0]
assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == ("NY", 660, 900, -1), "not the frozen EURUSD-02 rule"
print(f"{CID}: {F.slot_label('NY', 660)} -> {F.slot_label('NY', 900)}, short")

## 1. Data complete?

In [ ]:
cov = F.file_coverage(PAIR, START, END)
print(f"{PAIR} 2026: {cov['present']}/{cov['expected']} day files ({cov['share']:.2%}); missing {[str(d) for d in cov['missing']]}")
assert cov["share"] >= cfg.min_file_share, "download not complete: finish it before the one look"
panels = F.load_panels([PAIR], START, END, "holdout", cfg, clocks=("NY",))

## 2. The one look

In [ ]:
G, M, P = F.structure_detail(cand, START, END, "holdout", cfg, panels=panels, with_price=True)
comm_bps = F.commission_bps_per_lot(PAIR, COMMISSION_PER_LOT)
gross_pips = F.bps_to_pips(G[CID], P[CID], PAIR)
comm_pips = F.bps_to_pips(comm_bps, P[CID], PAIR)
net = pd.Series(gross_pips - comm_pips, index=G.index)
mae = pd.Series(F.bps_to_pips(M[CID], P[CID], PAIR), index=G.index)
m, se, t = (float(v[0]) for v in F.nw_tstats(net.to_numpy()[:, None], cfg.nw_lags))
p_pos = float(norm.sf(m / se))
p_below = float(norm.cdf((m - PROMISED_PIPS) / se))
H1, H2, H3 = m > 0, p_below <= 0.05, p_pos <= 0.05
verdict = "FAILS" if H2 else ("HOLDS" if H1 else "INCONCLUSIVE")
res = {"days": int(len(net)), "mean_net_pips": m, "se_pips": se, "t": m / se, "p_mean_gt_0": p_pos,
       "p_below_promised": p_below, "mean_pips_after_spread": float(np.mean(gross_pips)), "commission_pips": float(np.mean(comm_pips)),
       "win_rate": float((net > 0).mean()), "usd_per_lot_trade": float(np.mean((gross_pips - comm_pips) * F.usd_per_pip_per_lot(PAIR, P[CID]))),
       "typical_swing_pips": float(net.std()), "worst_open_loss_pips": float(mae.min()),
       "H1": bool(H1), "H2": bool(H2), "H3": bool(H3), "verdict": verdict}
display(pd.Series(res).to_frame("2026"))

## 3. By month, and 2024–2026 side by side

In [ ]:
monthly = net.groupby(net.index.to_period("M")).agg(["count", "mean", "sum"]).rename(columns={"count": "trades", "mean": "net pips/trade", "sum": "net pips"})
display(monthly.round(2)); monthly.to_csv(OUT / "05_monthly.csv")
prev = None
if not SYNTHETIC:
    Gp, Mp, Pp = F.structure_detail(cand, *F.CONFIRMATION, "confirmation", cfg, with_price=True)
    prev = pd.Series(F.bps_to_pips(Gp[CID], Pp[CID], PAIR) - F.bps_to_pips(comm_bps, Pp[CID], PAIR), index=Gp.index)
    yearly = pd.concat([prev, net]).groupby(lambda d: d.year).agg(["count", "mean"]).rename(columns={"count": "trades", "mean": "net pips/trade"})
    display(yearly.round(2)); yearly.to_csv(OUT / "05_yearly.csv")
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(net.index, net.cumsum(), label="2026 cumulative net pips")
ax.plot(net.index, PROMISED_PIPS * np.arange(1, len(net) + 1), "k--", lw=0.8, label=f"+{PROMISED_PIPS} pips/trade (2021-2025)")
ax.axhline(0, color="black", lw=0.5); ax.legend(); ax.set_ylabel("pips")
plt.tight_layout(); plt.savefig(OUT / "05_cumulative.png", dpi=110); plt.show()

## 4. A $6,000 prop account through 2026 (fixed size, 5% daily / 10% total limits)

In [ ]:
rows = []
for L in (2, 5):
    o, d, e = F.run_prop_path(G[[CID]].to_numpy(), M[[CID]].to_numpy(), np.array([comm_bps]), np.array([float(L)]), E0,
                              F.PropRules(0.05, 0.10, None, 0), len(G), compound=False)
    rows.append({"L": L, "lots": L * E0 / 100_000 / 1.10, "outcome": o, "day": d if o != "open" else None, "end_equity": round(e, 0)})
display(pd.DataFrame(rows))

In [ ]:
summary = {"study": "FXS05", "candidate": CID, "synthetic": SYNTHETIC, "window": [str(START), str(END)],
           "files": {k: cov[k] for k in ("expected", "present", "share")}, **res, "prop_6k": rows}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("days", "mean_net_pips", "t", "p_below_promised", "verdict")}, indent=2, default=str))